In [ ]:
# ── CRITICAL: Preprocessing anchor — MUST match fine-tuning window ──
# All climatology removal and std scaling is anchored to this period.
# Both obs and CMIP6 data are normalized relative to the SAME obs-derived
# statistics so the model always operates in obs-distribution space from
# pre-training onward. Changing this window INVALIDATES all saved checkpoints.

In [ ]:
import os, random, warnings, sys, signal
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as grad_ckpt
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import gc
import pickle
from tqdm import tqdm
import multiprocessing as mp

# ============================================================
# Configuration — Hybrid Settings
# ============================================================
SEED = 42

# Thread control: allow override via env var for flexibility
os.environ['OMP_NUM_THREADS'] = os.getenv('OMP_NUM_THREADS', '1')
os.environ['MKL_NUM_THREADS'] = os.getenv('MKL_NUM_THREADS', '1')
torch.set_num_threads(int(os.getenv('TORCH_NUM_THREADS', '1')))
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

# Multiprocessing: default to 0 for robustness, but configurable
if __name__ == '__main__':
    try:
        mp.set_start_method('spawn', force=True)
    except RuntimeError:
        pass


CLIMATOLOGY_PERIOD = slice('1960', '1990')
# ──────────────────────────────────────────────────────────────────────

CMIP6_START = 1870
CMIP6_END = 2014
OBS_START = 1960
OBS_END = 2024
PRETRAIN_VAL_START = 1994
FT_START = 1960
FT_END = 1990
TEST_START = 2000
TEST_END = OBS_END

OCEAN_VARS = ['sst', 'ohc300', 'ssh', 'siconc']
ATMOS_VARS = ['slp', 'tauu', 'tauv']
ALL_VARS = OCEAN_VARS + ATMOS_VARS

INPUT_WINDOW = 36
OUTPUT_SEQ_LEN = 24
SAMPLE_GAP = INPUT_WINDOW + OUTPUT_SEQ_LEN
BATCH_SIZE = 16
PRETRAIN_EPOCHS = 15
ACCUMULATION_STEPS = 2

# Model architecture
NHEAD = 4
DROPOUT = 0.1
NUM_LAYERS = 2
EMBED_DIM = 64

# Regularization & uncertainty
TIMESTEP_DROP_PROB = 0.1
MC_DROPOUT_PASSES = int(os.getenv('MC_PASSES', '15'))  # Configurable
ATTN_POOL_HW = 8
N_FOURIER_HARMONICS = 2

# Data loading: robust defaults, configurable
NUM_WORKERS = int(os.getenv('NUM_WORKERS', '0'))  # 0 = most robust
PIN_MEMORY = os.getenv('PIN_MEMORY', 'False').lower() == 'true'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training options
SCHEDULER_TYPE = os.getenv('SCHEDULER', 'warmup_cosine')  # 'warmup_cosine' or 'plateau'
ENABLE_TENSORBOARD = os.getenv('TENSORBOARD', 'True').lower() == 'true'
RESUME_TRAINING = os.getenv('RESUME', 'False').lower() == 'true'

# Spatial downsampling for memory efficiency
SPATIAL_DOWNSAMPLE_FACTOR = 2

# Output directories
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
TENSORBOARD_DIR = Path("runs")
TENSORBOARD_DIR.mkdir(exist_ok=True)

print(f"Using device: {DEVICE}")
print(f"NUM_WORKERS: {NUM_WORKERS} (override with env var)")
print(f"Scheduler: {SCHEDULER_TYPE}")
print(f"Input channels: {len(ALL_VARS)} → {ALL_VARS}")
print(f"MC Dropout passes: {MC_DROPOUT_PASSES}")


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()


def worker_init_fn(worker_id):
    """Initialize worker for DataLoader with proper seeding."""
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)
    torch.manual_seed(SEED + worker_id)
    # Ignore SIGINT in workers to prevent hangs
    signal.signal(signal.SIGINT, signal.SIG_IGN)


# ============================================================
# Linear Detrend Helper — IMPROVED: float precision
# ============================================================
def linear_detrend(da, ref_period):
    """Linear detrend using a fit computed from ref_period only."""
    da_ref = da.sel(time=ref_period)
    if len(da_ref.time) <= 1:
        return da
    ref_start = da_ref.time.values[0]
    # Use float days for sub-day precision (more accurate than .dt.days)
    def time_to_float_days(time_array, ref_start):
        try:
            return ((time_array - ref_start).dt.total_seconds() / 86400).astype(np.float64)
        except:
            # fallback for cftime
            return np.arange(len(time_array), dtype=np.float64)
    #t_num_ref = ((da_ref.time - ref_start).dt.total_seconds() / 86400).astype(np.float64)
    t_num_ref = time_to_float_days(da_ref.time, ref_start)
    da_ref_num = da_ref.assign_coords(time=t_num_ref)
    da_ref_num = da_ref_num.where(np.isfinite(da_ref_num))
    p = da_ref_num.polyfit(dim='time', deg=1, skipna=True)
    intercept = p.polyfit_coefficients.sel(degree=0, drop=True)
    slope = p.polyfit_coefficients.sel(degree=1, drop=True)
    #t_num_full = ((da.time - ref_start).dt.total_seconds() / 86400).astype(np.float64)
    t_num_full = time_to_float_days(da.time, ref_start)
    fit = intercept + slope * t_num_full
    return da - fit


# ============================================================
# Ocean Heat Content Integration
# ============================================================
def integrate_upper_ocean(da, max_depth_m=300):
    """Integrate temperature over the upper ocean (K·m)."""
    if 'lev' not in da.dims:
        return da
    depth = da.lev.values
    valid_idx = depth <= max_depth_m
    if not np.any(valid_idx):
        print(f" Warning: no levels within {max_depth_m}m; using shallowest")
        return da.isel(lev=0) * float(depth[0])
    depths_subset = depth[valid_idx]
    if len(depths_subset) == 1:
        return da.isel(lev=int(np.where(valid_idx)[0][0])) * float(depths_subset[0])
    mid = np.zeros(len(depths_subset) + 1)
    mid[0] = 0.0
    mid[1:-1] = (depths_subset[:-1] + depths_subset[1:]) / 2.0
    mid[-1] = min(depths_subset[-1] + (depths_subset[-1] - depths_subset[-2]) / 2.0, max_depth_m)
    dz = np.diff(mid)
    weights = xr.DataArray(dz, dims=['lev'], coords={'lev': depths_subset})
    integrated = (da.isel(lev=valid_idx) * weights).sum(dim='lev')
    return integrated


# ============================================================
# Wind Stress Calculation
# ============================================================
def calculate_wind_stress(u10, v10, rho_air=1.225):
    """τ = ρ_air × Cd × |U| × U (N/m²)."""
    
    wind_speed = np.sqrt(u10**2 + v10**2).fillna(0)

    # Variable drag coefficient (Large & Pond approximation)
    cd = 1.1e-3 + 0.035e-3 * wind_speed

    tauu = rho_air * cd * wind_speed * u10
    tauv = rho_air * cd * wind_speed * v10

    return tauu, tauv

# ============================================================
# Shared Normalization — IMPROVED: safer merge logic
# ============================================================
_OBS_ANCHOR_CLIM: dict | None = None
_OBS_ANCHOR_STD: dict | None = None


def set_obs_anchor(clim, std):
    """Register obs climatology as the global shared normalization anchor."""
    global _OBS_ANCHOR_CLIM, _OBS_ANCHOR_STD
    _OBS_ANCHOR_CLIM = clim
    _OBS_ANCHOR_STD = std
    print(f" ✓ Shared normalization anchor set "
          f"(period: {CLIMATOLOGY_PERIOD.start}–{CLIMATOLOGY_PERIOD.stop})")
    print(f" Variables anchored: {list(std.data_vars)}")


def compute_monthly_climatology(ds, ref_period=CLIMATOLOGY_PERIOD):
    """Monthly mean and std computed within ref_period."""
    ds = ds.sortby('time')
    ref = ds.sel(time=ref_period)
    clim = ref.groupby('time.month').mean('time')
    std = ref.groupby('time.month').std('time') + 1e-6
    return clim, std


def apply_preprocessing_shared(ds, own_clim, own_std, use_anchor=False):
    """Detrend → anomaly → standardize."""
    ds = ds.sortby('time')

    # 1. Detrend FIRST
    ds_dt = ds.map(lambda da: linear_detrend(da, CLIMATOLOGY_PERIOD))

    # 2. Compute anomalies
    ds_anom = ds_dt.groupby('time.month') - own_clim

    # 3. Standardize
    if use_anchor and _OBS_ANCHOR_STD is not None:
        anchor_std = _OBS_ANCHOR_STD

        shared_vars = [v for v in ds_anom.data_vars if v in anchor_std.data_vars]
        own_only = [v for v in ds_anom.data_vars if v not in anchor_std.data_vars]

        parts = []
        if shared_vars:
            parts.append(ds_anom[shared_vars].groupby('time.month') / anchor_std[shared_vars])
        if own_only:
            parts.append(ds_anom[own_only].groupby('time.month') / own_std[own_only])
        if not parts:
            raise ValueError("No variables available for standardization.")

        ds_std = xr.merge(parts) if len(parts) > 1 else parts[0]
    else:
        ds_std = ds_anom.groupby('time.month') / own_std

     # 4. Fill NaNs
    for var in ds_std.data_vars:
        #ds_std[var] = ds_std[var].fillna(ds_std[var].mean())
        ds_std[var] = ds_std[var].groupby('time.month').apply(
            lambda x: x.fillna(x.mean())
        )
    return ds_std


def apply_preprocessing(ds, clim, std):
    """Convenience wrapper for obs preprocessing."""
    return apply_preprocessing_shared(ds, clim, std, use_anchor=False)


def to_channel_da(ds_stdized, var_order=ALL_VARS):
    """Convert dataset to channel-first DataArray."""
    available = [v for v in var_order if v in ds_stdized.data_vars]
    ds_ordered = ds_stdized[available]
    for dim in list(ds_ordered.dims):
        if ds_ordered.sizes[dim] == 1 and dim not in ['time', 'lat', 'lon']:
            ds_ordered = ds_ordered.squeeze(dim, drop=True)
    da = ds_ordered.to_array('channel').transpose('time', 'channel', 'lat', 'lon')
    print(f" → shape: {da.shape} | channels: {available}")
    return da, available


# ============================================================
# Monthly IOD (Dipole Mode Index)
# ============================================================
def calculate_monthly_dmi(sst_data, base_period=CLIMATOLOGY_PERIOD):
    """Monthly DMI: West (50-70°E, 10°S-10°N) minus East (90-110°E, 10°S-0°N)."""
    if 'lev' in sst_data.dims:
        sst_data = sst_data.isel(lev=0)
    west = sst_data.sel(lat=slice(-10, 10), lon=slice(50, 70))
    east = sst_data.sel(lat=slice(-10, 0), lon=slice(90, 110))
    w_sst = west.weighted(np.cos(np.deg2rad(west.lat))).mean(['lat', 'lon'])
    e_sst = east.weighted(np.cos(np.deg2rad(east.lat))).mean(['lat', 'lon'])
    raw_dmi = w_sst - e_sst
    w_clim = w_sst.sel(time=base_period).groupby('time.month').mean('time')
    e_clim = e_sst.sel(time=base_period).groupby('time.month').mean('time')
    dmi_anom = raw_dmi.groupby('time.month') - (w_clim - e_clim)
    dmi_anom = linear_detrend(dmi_anom, base_period)
    return dmi_anom


# ============================================================
# Calendar Helper
# ============================================================
def convert_to_standard_calendar(data):
    """Convert non-standard (e.g. cftime) calendar to standard datetime64."""
    if len(data.time) == 0:
        return data
    first = data.time.values[0]
    if isinstance(first, (np.datetime64, pd.Timestamp)):
        return data
    if not hasattr(first, 'year'):
        return data
    time_pd = pd.DatetimeIndex([
        pd.Timestamp(t.year, t.month, t.day) for t in data.time.values
    ])
    if not isinstance(data, xr.DataArray):
        raise TypeError("convert_to_standard_calendar expects DataArray")
        
    return xr.DataArray(
        data.values, dims=data.dims,
        coords={**{k: v for k, v in data.coords.items() if k != 'time'},
                'time': time_pd},
        name=data.name, attrs=data.attrs
    )


def ensure_datetime_slice(da, time_slice):
    """Helper to safely slice with string dates after conversion."""
    if hasattr(da.time.values[0], 'year'):
        da = convert_to_standard_calendar(da)
    return da.sel(time=time_slice)


# ============================================================
# Data Loading — CMIP6
# ============================================================
def get_cmip6_model_files(base_path="model_1x1/"):
    base_path = Path(base_path)
    nc_files = [f for f in base_path.glob("*.nc") if '.ipynb_checkpoints' not in str(f)]
    model_files = {}
    for f in nc_files:
        model_files.setdefault(f.stem.split('_')[0], []).append(f)
    print(f"Found {len(model_files)} CMIP6 models")
    return model_files


def load_single_cmip6_model(base_path, model_name, model_files_dict):
    """Load CMIP6 model data, computing wind stress from uas/vas if needed."""
    print(f" Loading CMIP6: {model_name}")
    files = model_files_dict.get(model_name, [])
    if not files:
        raise ValueError(f"No files for model {model_name}")
    t_slice = slice(str(CMIP6_START), str(CMIP6_END))
    loaded = {}
    
    var_map = {
        'sst': 'tos',
        'ohc300': 'thetao',
        'ssh': 'zos',
        'siconc': 'siconc',
        'slp': 'psl',
    }
    
    for target_var, cmip_var in var_map.items():
        f = next((f for f in files if cmip_var in f.stem), None)
        if not f:
            print(f"   Warning: No file for {cmip_var} in {model_name}")
            continue
        #ds = xr.open_dataset(f)
        ds = xr.open_dataset(f, chunks={'time': 50})
        actual_var = next((v for v in ds.data_vars if cmip_var in v), None)
        if actual_var is None:
            print(f"   Warning: {cmip_var} not found in {f.name}")
            ds.close()
            continue
        da = convert_to_standard_calendar(ds[actual_var])
        if cmip_var == 'thetao':
            da = integrate_upper_ocean(da)
        elif 'lev' in da.dims:
            da = da.isel(lev=0) if da.sizes['lev'] == 1 else da.mean('lev')
        loaded[target_var] = ensure_datetime_slice(da, t_slice)
        ds.close()
        print(f"   ✓ Loaded {target_var} from {cmip_var}")
    
    # Compute wind stress from uas/vas
    uas_file = next((f for f in files if 'uas' in f.stem), None)
    vas_file = next((f for f in files if 'vas' in f.stem), None)
    
    if uas_file and vas_file:
        print(f"   Computing wind stress from uas/vas for {model_name}")
        ds_uas = xr.open_dataset(uas_file)
        ds_vas = xr.open_dataset(vas_file)
        
        uas_var = next((v for v in ds_uas.data_vars if 'uas' in v.lower()), None)
        vas_var = next((v for v in ds_vas.data_vars if 'vas' in v.lower()), None)
        
        if uas_var and vas_var:
            u10 = convert_to_standard_calendar(ds_uas[uas_var])
            v10 = convert_to_standard_calendar(ds_vas[vas_var])
            tauu, tauv = calculate_wind_stress(
                ensure_datetime_slice(u10, t_slice),
                ensure_datetime_slice(v10, t_slice)
            )
            loaded['tauu'] = tauu
            loaded['tauv'] = tauv
            ds_uas.close()
            ds_vas.close()
            print(f"   ✓ Wind stress computed successfully")
        else:
            print(f"   ⚠ Could not find uas/vas variables in files")
            ds_uas.close()
            ds_vas.close()
    else:
        print(f"   ⚠ No uas/vas files found for {model_name}, wind stress will be missing")
    
    if 'sst' not in loaded:
        raise ValueError(f"Model {model_name}: SST is required")
    
    expected_vars = ['sst', 'ohc300', 'ssh', 'siconc', 'slp', 'tauu', 'tauv']
    available_vars = [v for v in expected_vars if v in loaded]
    print(f" ✓ {model_name} — loaded {len(available_vars)}/{len(expected_vars)} variables: {available_vars}")

    #EXPECTED_VARS = ['sst', 'ohc300', 'ssh', 'siconc', 'slp', 'tauu', 'tauv']

    missing = [v for v in expected_vars if v not in loaded]
    
    if missing:
        print(f" ✗ Skipping {model_name} due to missing variables: {missing}")
        return None
        
    return xr.Dataset(loaded)



# ============================================================
# Data Loading — Observations
# ============================================================
def load_obs_data(iod_only=False):
    if iod_only:
        obs_path = Path("obs_1x1/sst_obs/")
        sst_files = list(obs_path.glob("*.nc"))
        if not sst_files:
            raise FileNotFoundError(f"No SST files in {obs_path}")
        ds = xr.open_dataset(sst_files[0])
        sst_var = next((v for v in ['sst', 'tos', 'sst_monthly', 'sea_surface_temperature']
                        if v in ds.data_vars), list(ds.data_vars)[0])
        sst = convert_to_standard_calendar(ds[sst_var])
        ds.close()
        return xr.Dataset({'sst': sst})
    
    obs_path = Path("obs_1x1/obs/")
    if not obs_path.exists():
        raise FileNotFoundError(f"Obs path not found: {obs_path}")
    t_slice = slice(str(OBS_START), str(OBS_END))
    obs_data = {}
    patterns = {
        'sst': (['*sst*.nc', '*tos*.nc'], ['sst', 'tos', 'sea_surface_temperature']),
        'ohc300': (['*sohtc300*.nc', '*ohc*.nc', '*thetao*.nc'], ['sohtc', 'ohc', 'thetao']),
        'ssh': (['*sossheig*.nc', '*zos*.nc', '*ssha*.nc', '*ssh*.nc'], ['sossheig', 'zos', 'ssha', 'ssh']),
        'siconc': (['*siconc*.nc', '*sic*.nc', '*sea_ice*.nc'], ['siconc', 'sic', 'ice']),
        'slp': (['*msl*.nc', '*psl*.nc', '*slp*.nc'], ['msl', 'psl', 'slp']),
    }
    for var, (patterns_list, candidates) in patterns.items():
        f = next((f for pat in patterns_list for f in obs_path.glob(pat)), None)
        if not f:
            continue
        ds = xr.open_dataset(f)
        actual_var = next((v for v in ds.data_vars
                           if any(c in v.lower() for c in candidates)),
                          list(ds.data_vars)[0])
        da = convert_to_standard_calendar(ds[actual_var])
        if var == 'ohc300' and 'lev' in da.dims:
            da = integrate_upper_ocean(da)
        elif 'lev' in da.dims:
            da = da.isel(lev=0) if da.sizes['lev'] == 1 else da.mean('lev')
        obs_data[var] = ensure_datetime_slice(da, t_slice)
        ds.close()
        print(f" ✓ {var} from {f.name}")
    
    u_f = next((f for pat in ['*u10*.nc', '*uas*.nc'] for f in obs_path.glob(pat)), None)
    v_f = next((f for pat in ['*v10*.nc', '*vas*.nc'] for f in obs_path.glob(pat)), None)
    if u_f and v_f:
        ds_u = xr.open_dataset(u_f)
        ds_v = xr.open_dataset(v_f)
        u_var = next((v for v in ds_u.data_vars if 'u10' in v.lower() or 'uas' in v.lower()),
                     list(ds_u.data_vars)[0])
        v_var = next((v for v in ds_v.data_vars if 'v10' in v.lower() or 'vas' in v.lower()),
                     list(ds_v.data_vars)[0])
        u10 = convert_to_standard_calendar(ds_u[u_var])
        v10 = convert_to_standard_calendar(ds_v[v_var])
        obs_data['tauu'], obs_data['tauv'] = calculate_wind_stress(
            ensure_datetime_slice(u10, t_slice),
            ensure_datetime_slice(v10, t_slice)
        )
        ds_u.close(); ds_v.close()
        print(f" ✓ Wind stress computed from {u_f.name}, {v_f.name}")
    else:
        combined = list(obs_path.glob("*wind*.nc")) + list(obs_path.glob("*uv10*.nc"))
        if combined:
            ds_w = xr.open_dataset(combined[0])
            u_var = next((v for v in ds_w.data_vars if 'u10' in v.lower()), None)
            v_var = next((v for v in ds_w.data_vars if 'v10' in v.lower()), None)
            if u_var and v_var:
                u10 = convert_to_standard_calendar(ds_w[u_var])
                v10 = convert_to_standard_calendar(ds_w[v_var])
                obs_data['tauu'], obs_data['tauv'] = calculate_wind_stress(
                    ensure_datetime_slice(u10, t_slice),
                    ensure_datetime_slice(v10, t_slice)
                )
                print(f" ✓ Wind stress from {combined[0].name}")
            ds_w.close()
    
    missing = [v for v in ALL_VARS if v not in obs_data]
    if missing:
        raise ValueError(f"Missing obs variables: {missing}")
    return xr.Dataset(obs_data)
    
# ============================================================
# Dataset split- helper
# ============================================================
def split_dataset_by_time(dataset, train_end_year, val_start_year):
    train_idx = []
    val_idx = []

    for i, t in enumerate(dataset.sample_target_times):
        # ensure FULL output sequence stays in train period
        if t.year + (OUTPUT_SEQ_LEN // 12) <= train_end_year:
            train_idx.append(i)
        elif t.year >= val_start_year:
            val_idx.append(i)

    return (
        torch.utils.data.Subset(dataset, train_idx),
        torch.utils.data.Subset(dataset, val_idx)
    )


# ============================================================
# Dataset — IMPROVED: configurable noise decay
# ============================================================
class IODDataset(Dataset):
    """Sequence prediction dataset with optional augmentation."""
    def __init__(self, da, iod_norm, channel_names,
                 target_start_year=None, target_end_year=None,
                 augment=False, noise_scale=0.01, noise_decay_epoch=None):
        self.time = pd.to_datetime(da.time.values)
        self.data = da.values.astype(np.float32)
        self.iod_norm = iod_norm.values.astype(np.float32)
        self.channel_names = channel_names
        self.augment = augment
        self.noise_scale = noise_scale
        self.noise_decay_epoch = noise_decay_epoch
        self._current_epoch = 0  # For noise decay tracking
        self.samples = []
        self.sample_target_times = []
        self.sample_months = []
        self._build_samples(target_start_year, target_end_year)

    def _build_samples(self, target_start_year=None, target_end_year=None):
        n = len(self.time)
        for t in range(n - INPUT_WINDOW - OUTPUT_SEQ_LEN + 1):
            target_start_idx = t + INPUT_WINDOW
            target_time = self.time[target_start_idx]
            if target_start_year is not None and target_time.year < target_start_year:
                continue
            if target_end_year is not None and target_time.year > target_end_year:
                continue
            self.samples.append((t, target_start_idx, target_start_idx + OUTPUT_SEQ_LEN))
            self.sample_target_times.append(target_time)
            self.sample_months.append([self.time[t + i].month - 1 for i in range(INPUT_WINDOW)])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        start_idx, target_start_idx, target_end_idx = self.samples[i]
        X = torch.from_numpy(self.data[start_idx:start_idx + INPUT_WINDOW].copy())
        
        if self.augment:
            current_noise = self.noise_scale
            if self.noise_decay_epoch is not None and self.noise_decay_epoch > 0:
                decay_factor = max(0.1, 1.0 - self._current_epoch / self.noise_decay_epoch)
                current_noise *= decay_factor
        
            # multiplicative noise
            X = X * (1 + torch.randn_like(X) * current_noise)
        
            # channel dropout
            if torch.rand(1) < 0.1:
                drop_idx = torch.randint(0, X.shape[1], (1,))
                X[:, drop_idx, :, :] = 0
            
        y = torch.from_numpy(self.iod_norm[target_start_idx:target_end_idx].copy())
        months = torch.tensor(self.sample_months[i], dtype=torch.long)
        return X, y, months
    
    def set_epoch(self, epoch):
        """Update epoch for noise decay scheduling."""
        self._current_epoch = epoch


def create_safe_dataloader(dataset, batch_size, shuffle=True, drop_last=False):
    """Create a DataLoader with robust settings."""
    return DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=drop_last,
        worker_init_fn=worker_init_fn if NUM_WORKERS > 0 else None
    )


# ============================================================
# Fourier Seasonal Encoding
# ============================================================
class FourierSeasonalEncoding(nn.Module):
    """Periodic month encoding via Fourier features + small learned residual."""
    def __init__(self, embed_dim=EMBED_DIM, n_harmonics=N_FOURIER_HARMONICS,
                 learnable_dim=None):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_harmonics = n_harmonics
        fourier_dim = 2 * n_harmonics
        learnable_dim = learnable_dim or (embed_dim // 4)
        self.learnable = nn.Embedding(12, learnable_dim)
        self.proj = nn.Linear(fourier_dim + learnable_dim, embed_dim, bias=False)
        months = torch.arange(12, dtype=torch.float32)
        feats = []
        for k in range(1, n_harmonics + 1):
            feats.append(torch.sin(2 * torch.pi * k * months / 12))
            feats.append(torch.cos(2 * torch.pi * k * months / 12))
        fourier_table = torch.stack(feats, dim=1)
        self.register_buffer('fourier_table', fourier_table)

    def forward(self, months):
        fourier = self.fourier_table[months]
        residual = self.learnable(months)
        combined = torch.cat([fourier, residual], dim=-1)
        return self.proj(combined)


# ============================================================
# Model Architecture — HYBRID: granular checkpointing
# ============================================================
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_c, out_c, kernel=3, padding=1):
        super().__init__()
        self.dw = nn.Conv2d(in_c, in_c, kernel, padding=padding, groups=in_c, bias=False)
        self.pw = nn.Conv2d(in_c, out_c, 1, bias=False)
    def forward(self, x):
        return self.pw(self.dw(x))


class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, spatial_hw=(ATTN_POOL_HW, ATTN_POOL_HW)):
        super().__init__()
        self.attn_conv = nn.Conv2d(in_channels, 1, kernel_size=1, bias=True)
        self.spatial_hw = spatial_hw

    def forward(self, x):
        B, C, H, W = x.shape
        logits = self.attn_conv(x)
        weights = torch.softmax(logits.view(B, 1, -1), dim=-1).view(B, 1, H, W)
        pooled = (x * weights).sum(dim=(-2, -1))
        return pooled


class SpatialCNN(nn.Module):
    """Spatial encoder with GRANULAR gradient checkpointing per stage."""
    def __init__(self, in_channels, out_dim=EMBED_DIM, pool_hw=ATTN_POOL_HW,
                 downsample_factor=SPATIAL_DOWNSAMPLE_FACTOR):
        super().__init__()
        self.downsample = nn.AvgPool2d(downsample_factor) if downsample_factor > 1 else nn.Identity()
        self.conv1 = nn.Conv2d(in_channels, 32, 3, padding=1)
        self.conv2 = DepthwiseSeparableConv(32, 64)
        self.conv3 = DepthwiseSeparableConv(64, 128)
        self.upsample = nn.AdaptiveAvgPool2d((pool_hw, pool_hw))
        self.attn_pool = AttentionPool2d(128, (pool_hw, pool_hw))
        self.fc = nn.Linear(128, out_dim)
        self.relu = nn.ReLU(inplace=True)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.dropout = nn.Dropout(DROPOUT)

    def _stage1(self, x): return self.relu(self.bn1(self.conv1(x)))
    def _stage2(self, x): return self.relu(self.bn2(self.conv2(x)))
    def _stage3(self, x): return self.relu(self.bn3(self.conv3(x)))

    def forward(self, x):
        x = self.downsample(x)
        # GRANULAR checkpointing: checkpoint each stage separately (memory efficient)
        if self.training:
            x = grad_ckpt.checkpoint(self._stage1, x, use_reentrant=False)
            x = grad_ckpt.checkpoint(self._stage2, x, use_reentrant=False)
            x = grad_ckpt.checkpoint(self._stage3, x, use_reentrant=False)
        else:
            # No checkpointing at inference for speed
            x = self._stage1(x)
            x = self._stage2(x)
            x = self._stage3(x)
        x = self.upsample(x)
        x = self.attn_pool(x)
        x = self.dropout(x)
        return self.fc(x)


class TemporalEncoder(nn.Module):
    def __init__(self, d_model, max_len=INPUT_WINDOW):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=NHEAD,
            dim_feedforward=4 * d_model, dropout=DROPOUT,
            batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(enc_layer, NUM_LAYERS)

    def forward(self, x):
        x = x + self.pos[:, :x.size(1), :]
        return self.encoder(x)


class HeteroscedasticHead(nn.Module):
    LOG_VAR_MIN = -10.0
    LOG_VAR_MAX = 5.0

    def __init__(self, embed_dim):
        super().__init__()
        hidden = embed_dim // 2
        self.shared = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )
        self.mean_head = nn.Linear(hidden, 1)
        self.log_var_head = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.shared(x)
        mean = self.mean_head(h).squeeze(-1)
        log_var = self.log_var_head(h).squeeze(-1)
        log_var = torch.clamp(log_var, self.LOG_VAR_MIN, self.LOG_VAR_MAX)
        return mean, log_var


def gaussian_nll_loss(mean, log_var, target):
    precision = torch.exp(-log_var)
    nll = 0.5 * (log_var + (target - mean).pow(2) * precision)
    return nll.mean()


class SpatioTemporalIOD(nn.Module):
    def __init__(self, num_channels, embed_dim=EMBED_DIM, out_seq=OUTPUT_SEQ_LEN):
        super().__init__()
        self.spatial = SpatialCNN(num_channels, embed_dim)
        self.season_enc = FourierSeasonalEncoding(embed_dim)
        self.encoder = TemporalEncoder(embed_dim)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=NHEAD,
            dim_feedforward=4 * embed_dim, dropout=DROPOUT,
            batch_first=True, activation='gelu'
        )
        self.decoder = nn.TransformerDecoder(dec_layer, NUM_LAYERS)
        self.query_embed = nn.Parameter(torch.randn(1, out_seq, embed_dim) * 0.02)
        self.head = HeteroscedasticHead(embed_dim)
        mask = torch.triu(torch.ones(out_seq, out_seq), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        self.register_buffer("causal_mask", mask)
        mask = self.causal_mask.to(x.device)

    def forward(self, x, months, use_checkpoint=False):
        B, T, C, H, W = x.shape
        if self.training and TIMESTEP_DROP_PROB > 0:
            keep = (torch.rand(B, T, 1, 1, 1, device=x.device) > TIMESTEP_DROP_PROB).float()
            x = x * keep
        #x_flat = x.reshape(B * T, C, H, W)
        x_flat = x.contiguous().view(B * T, C, H, W)
        spatial_features = self.spatial(x_flat).view(B, T, -1)
        spatial_features = spatial_features + self.season_enc(months)
        
        # Optional checkpointing at encoder level (for very large models)
        if use_checkpoint and self.training:
            memory = grad_ckpt.checkpoint(self.encoder, spatial_features, use_reentrant=False)
        else:
            memory = self.encoder(spatial_features)
            
        queries = self.query_embed.expand(B, -1, -1)
        decoder_out = self.decoder(tgt=queries, memory=memory, tgt_mask=self.causal_mask)
        return self.head(decoder_out)
      
    def predict_with_uncertainty(self, x, months, n_passes=MC_DROPOUT_PASSES):
        self.eval()
    
        # Enable dropout only
        def enable_dropout(m):
            if isinstance(m, nn.Dropout):
                m.train()
    
        self.apply(enable_dropout)
    
        means_list = []
        log_vars_list = []
    
        with torch.no_grad():
            for _ in range(n_passes):
                B, T, C, H, W = x.shape
                x_flat = x.reshape(B * T, C, H, W)
                sp_feat = self.spatial(x_flat).view(B, T, -1)
                sp_feat = sp_feat + self.season_enc(months)
    
                memory = self.encoder(sp_feat)
                queries = self.query_embed.expand(B, -1, -1)
    
                dec_out = self.decoder(
                    tgt=queries,
                    memory=memory,
                    tgt_mask=self.causal_mask
                )
    
                mu, lv = self.head(dec_out)
                means_list.append(mu)
                log_vars_list.append(lv)
    
        means = torch.stack(means_list, dim=0)
        log_vars = torch.stack(log_vars_list, dim=0)
    
        mean_pred = means.mean(dim=0)
        sigma_epistemic = means.std(dim=0)
        sigma_aleatoric = torch.exp(0.5 * log_vars.mean(dim=0))
        sigma_total = torch.sqrt(sigma_aleatoric**2 + sigma_epistemic**2)
    
        return mean_pred, sigma_aleatoric, sigma_epistemic, sigma_total

# ============================================================
# Learning Rate Schedulers — HYBRID: configurable
# ============================================================
class WarmupCosineScheduler:
    """Warmup + cosine decay scheduler (better for transformers)."""
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr, min_lr=0):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_epoch = 0
        
    def step(self, epoch=None):
        if epoch is None:
            epoch = self.current_epoch
        self.current_epoch = epoch + 1
        
        if epoch < self.warmup_epochs:
            lr = self.base_lr * (epoch + 1) / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + np.cos(np.pi * progress))
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr


def get_scheduler(optimizer, scheduler_type, **kwargs):
    """Factory function for configurable LR scheduling."""
    if scheduler_type == 'warmup_cosine':
        return WarmupCosineScheduler(optimizer, **kwargs)
    elif scheduler_type == 'plateau':
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, **kwargs)
    else:
        raise ValueError(f"Unknown scheduler type: {scheduler_type}")

def state_dict(self):
    return {"current_epoch": self.current_epoch}

def load_state_dict(self, state):
    self.current_epoch = state.get("current_epoch", 0)

# ============================================================
# Checkpoint Management — IMPROVED: resume capability
# ============================================================
def load_checkpoint_if_exists(model, optimizer, checkpoint_path, scheduler=None):
    """Load checkpoint if it exists for resuming training."""
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        if optimizer and 'optimizer_state_dict' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if scheduler and 'scheduler_state_dict' in checkpoint:
            try:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            except:
                print("⚠ Scheduler state not compatible, skipping.")            
        start_epoch = checkpoint.get('epoch', 0) + 1
        best_val = checkpoint.get('best_val', np.inf)
        print(f"✓ Resumed from checkpoint (epoch {start_epoch}, best_val={best_val:.4f})")
        return start_epoch, best_val
    return 0, np.inf


def save_checkpoint(model, optimizer, epoch, best_val, checkpoint_path, scheduler=None):
    """Save training checkpoint with full state."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val': best_val,
    }
    #if scheduler:
        #checkpoint['scheduler_state_dict'] = scheduler.state_dict()
    if scheduler is not None:
        checkpoint['scheduler_state_dict'] = scheduler.state_dict()
    torch.save(checkpoint, checkpoint_path)


# ============================================================
# Training Functions — HYBRID: TensorBoard + autocast + resume
# ============================================================
def compute_metrics(pred, target):
    pred = pred.flatten()
    target = target.flatten()
    
    valid = ~(np.isnan(pred) | np.isnan(target))
    pred, target = pred[valid], target[valid]
    
    if len(pred) < 2:
        return np.nan, np.nan
    
    rmse = np.sqrt(np.mean((pred - target) ** 2))
    corr = np.corrcoef(pred, target)[0, 1]
    return rmse, corr


def train_model(model, train_loader, valid_loader, epochs, lr,
                checkpoint_name, is_finetune=False, writer=None, resume=False):
    if is_finetune:
        optimizer = torch.optim.AdamW([
            {'params': model.encoder.parameters(), 'lr': lr * 0.1},
            {'params': model.spatial.parameters(), 'lr': lr * 0.1},
            {'params': model.season_enc.parameters(), 'lr': lr * 0.5},
            {'params': model.decoder.parameters(), 'lr': lr},
            {'params': model.head.parameters(), 'lr': lr},
            {'params': [model.query_embed], 'lr': lr},
        ], weight_decay=1e-3)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    # Configurable scheduler
    if SCHEDULER_TYPE == 'warmup_cosine':
        scheduler = get_scheduler(optimizer, SCHEDULER_TYPE, 
                                 warmup_epochs=2, total_epochs=epochs, 
                                 base_lr=lr, min_lr=lr * 0.01)
    else:  # plateau
        scheduler = get_scheduler(optimizer, SCHEDULER_TYPE,
                                 mode='min', patience=3, factor=0.5)

    scaler = GradScaler() if DEVICE == 'cuda' else None
    ckpt = CHECKPOINT_DIR / f"best_{checkpoint_name}.pt"
    resume_ckpt = CHECKPOINT_DIR / f"resume_{checkpoint_name}.pt"
    
    start_epoch = 0
    best_val = np.inf
    
    if resume:
        start_epoch, best_val = load_checkpoint_if_exists(model, optimizer, resume_ckpt, scheduler)
    
    for ep in range(start_epoch, epochs):
        model.train()
        # Update dataset epoch for noise decay
        if hasattr(train_loader.dataset, 'set_epoch'):
            train_loader.dataset.set_epoch(ep)
            
        if is_finetune:
            for module in model.modules():
                if isinstance(module, nn.modules.batchnorm._BatchNorm):
                    module.eval()
                    for p in module.parameters():
                        p.requires_grad = False

        tr_losses = []
        optimizer.zero_grad()
        
        if SCHEDULER_TYPE == 'warmup_cosine':
            current_lr = scheduler.step(ep)
        else:
            current_lr = optimizer.param_groups[0]['lr']

        for bi, (X, y, months) in enumerate(tqdm(train_loader, desc=f"Epoch {ep+1}", leave=False)):
            X, y, months = X.to(DEVICE), y.to(DEVICE), months.to(DEVICE)

            if DEVICE == 'cuda':
                with autocast(device_type='cuda'):
                    mean, log_var = model(X, months)
                    loss = gaussian_nll_loss(mean, log_var, y) / ACCUMULATION_STEPS
                scaler.scale(loss).backward()
            else:
                mean, log_var = model(X, months)
                loss = gaussian_nll_loss(mean, log_var, y) / ACCUMULATION_STEPS
                loss.backward()

            tr_losses.append(loss.item() * ACCUMULATION_STEPS)

            if (bi + 1) % ACCUMULATION_STEPS == 0 or (bi + 1 == len(train_loader)):
                if DEVICE == 'cuda':
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                optimizer.zero_grad()

        model.eval()
        val_losses = []
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for X, y, months in valid_loader:
                X, y, months = X.to(DEVICE), y.to(DEVICE), months.to(DEVICE)
                # IMPROVED: Use autocast in validation for consistency
                if DEVICE == 'cuda':
                    with autocast(device_type='cuda'):
                        mean, log_var = model(X, months)
                        val_losses.append(gaussian_nll_loss(mean, log_var, y).item())
                        val_preds.append(mean.cpu().numpy())
                        val_targets.append(y.cpu().numpy())
                else:
                    mean, log_var = model(X, months)
                    val_losses.append(gaussian_nll_loss(mean, log_var, y).item())
                    val_preds.append(mean.cpu().numpy())
                    val_targets.append(y.cpu().numpy())

        val_nll = np.mean(val_losses)
        val_preds = np.concatenate(val_preds)
        val_targets = np.concatenate(val_targets)
        
        val_rmse, val_corr = compute_metrics(val_preds, val_targets)
      
        if SCHEDULER_TYPE == 'plateau':
            scheduler.step(val_nll)
        
        # TensorBoard logging
        if writer and ENABLE_TENSORBOARD:
            writer.add_scalar(f'{checkpoint_name}/train_loss', np.mean(tr_losses), ep)
            writer.add_scalar(f'{checkpoint_name}/val_loss', val_nll, ep)
            writer.add_scalar(f'{checkpoint_name}/learning_rate', current_lr, ep)
            writer.add_scalar(f'{checkpoint_name}/val_rmse', val_rmse, ep)
            writer.add_scalar(f'{checkpoint_name}/val_corr', val_corr, ep)

        if val_nll < best_val:
            best_val = val_nll
            torch.save(model.state_dict(), ckpt)
            
        # Save resume checkpoint
        if RESUME_TRAINING:
            save_checkpoint(model, optimizer, ep, best_val, resume_ckpt, 
                          scheduler if SCHEDULER_TYPE == 'warmup_cosine' else None)

        print(f" Ep {ep+1:02d}/{epochs} | LR={current_lr:.6f} | "
          f"Train NLL={np.mean(tr_losses):.4f} | "
          f"Val NLL={val_nll:.4f} | RMSE={val_rmse:.4f} | Corr={val_corr:.3f}")

    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False))
        print(f" → Best checkpoint loaded (Val NLL={best_val:.4f})")
        
    if resume_ckpt.exists() and not RESUME_TRAINING:
        resume_ckpt.unlink()

    return model, best_val


# ============================================================
# ZERO-SHOT DIAGNOSTIC — From user's script (excellent addition!)
# ============================================================
def evaluate_zero_shot(model, eval_loader, channel_indices, device):
    """Evaluate pre-trained model on observation data before fine-tuning."""
    model.eval()
    losses = []
    with torch.no_grad():
        for X, y, months in eval_loader:
            # Subset channels to match model's pre-training
            X_subset = X[:, :, channel_indices, :, :].to(device)
            mu, lv = model(X_subset, months.to(device))
            losses.append(gaussian_nll_loss(mu, lv, y.to(device)).item())
    return np.mean(losses)


# ============================================================
# Inference and Evaluation — IMPROVED: consistent settings
# ============================================================
def run_inference(model, dataset, iod_mean, iod_std, return_uncertainty=False):
    model.eval()
    preds_list = []
    tgts_list = []
    sig_aleat_list = []
    sig_epist_list = []
    sig_total_list = []

    # Use consistent batch size and worker settings
    loader = create_safe_dataloader(dataset, batch_size=BATCH_SIZE//2, shuffle=False, 
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    with torch.no_grad():
        for X, y, months in tqdm(loader, desc="Inference", leave=False):
            X_d = X.to(DEVICE)
            m_d = months.to(DEVICE)
            if return_uncertainty:
                mu, s_al, s_ep, s_tot = model.predict_with_uncertainty(X_d, m_d)
                sig_aleat_list.append(s_al.cpu().numpy())
                sig_epist_list.append(s_ep.cpu().numpy())
                sig_total_list.append(s_tot.cpu().numpy())
            else:
                mu, _ = model(X_d, m_d)
            preds_list.append(mu.cpu().numpy())
            tgts_list.append(y.numpy())

    preds = np.concatenate(preds_list, axis=0) * iod_std + iod_mean
    tgts = np.concatenate(tgts_list, axis=0) * iod_std + iod_mean
    target_times = list(dataset.sample_target_times)

    if return_uncertainty:
        s_al = np.concatenate(sig_aleat_list, axis=0) * iod_std
        s_ep = np.concatenate(sig_epist_list, axis=0) * iod_std
        s_tot = np.concatenate(sig_total_list, axis=0) * iod_std
        return preds, tgts, target_times, s_al, s_ep, s_tot
    return preds, tgts, target_times


def murphy_skill_score(pred, obs):
    mse_model = np.mean((pred - obs) ** 2)
    mse_clim = np.mean((obs - obs.mean()) ** 2)
    return 1.0 - mse_model / (mse_clim + 1e-12)


def print_verification_table(preds, targets, tag="",
                             sig_aleat=None, sig_epist=None):
    print(f"\n{'='*90}")
    print(f" Verification — {tag}")
    hdr = f" {'Lead':>5} {'RMSE':>7} {'Corr':>7} {'MSS':>7} {'N':>6}"
    if sig_aleat is not None:
        hdr += f" {'σ_aleat':>8} {'σ_epist':>8}"
    print(hdr)
    print(f" {'-'*70}")
    for lead in [1, 3, 6, 9, 12, 18, 24]:
        if lead > OUTPUT_SEQ_LEN:
            continue
        li = lead - 1
        p, t = preds[:, li], targets[:, li]
        valid = ~(np.isnan(p) | np.isnan(t))
        p, t = p[valid], t[valid]
        if len(p) < 2:
            continue
        rmse = float(np.sqrt(np.mean((p - t)**2)))
        corr = float(np.corrcoef(p, t)[0, 1])
        mss = float(murphy_skill_score(p, t))
        row = f" {lead:>5}m {rmse:>7.4f} {corr:>7.3f} {mss:>7.3f} {len(p):>6}"
        if sig_aleat is not None:
            row += f" {np.mean(sig_aleat[valid, li]):>8.4f} {np.mean(sig_epist[valid, li]):>8.4f}"
        print(row)
    print(f"{'='*90}")


def save_timeseries_figure(target_times, targets, preds, tag, fname,
                           sig_total=None):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    leads = [1, 6, 12, 24]
    for idx, lead in enumerate(leads):
        ax = axes[idx // 2, idx % 2]
        li = lead - 1
        ax.plot(target_times, targets[:, li], label='Observed', linewidth=2)
        ax.plot(target_times, preds[:, li], '--', label=f'Pred (L{lead}m)', linewidth=1.8)
        if sig_total is not None:
            lo = preds[:, li] - sig_total[:, li]
            hi = preds[:, li] + sig_total[:, li]
            ax.fill_between(target_times, lo, hi, alpha=0.25, label='±1σ total')
        ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
        ax.set_title(f'Lead {lead} months — {tag}')
        ax.set_xlabel('Year'); ax.set_ylabel('IOD Index (°C)')
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = FIGURE_DIR / fname
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f' → Figure: {path}')


def plot_predictability_vs_lead(preds, targets, tag, fname,
                                sig_aleat=None, sig_epist=None):
    n_panels = 4 if sig_aleat is not None else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
    corr_list, mss_list = [], []
    for li in range(OUTPUT_SEQ_LEN):
        p, t = preds[:, li], targets[:, li]
        valid = ~(np.isnan(p) | np.isnan(t))
        if valid.sum() > 2:
            corr_list.append(np.corrcoef(p[valid], t[valid])[0, 1])
            mss_list.append(murphy_skill_score(p[valid], t[valid]))
        else:
            corr_list.append(np.nan); mss_list.append(np.nan)
    leads = range(1, OUTPUT_SEQ_LEN + 1)
    axes[0].plot(leads, corr_list, marker='o', linewidth=2)
    axes[0].set_xlabel('Lead (months)'); axes[0].set_ylabel('Correlation')
    axes[0].set_title(f'Correlation vs Lead — {tag}')
    axes[0].grid(True, alpha=0.3); axes[0].axhline(0, color='grey', linestyle='--')
    axes[1].plot(leads, mss_list, marker='s', linewidth=2, color='green')
    axes[1].set_xlabel('Lead (months)'); axes[1].set_ylabel('Murphy Skill Score')
    axes[1].set_title(f'Skill Score vs Lead — {tag}')
    axes[1].grid(True, alpha=0.3); axes[1].axhline(0, color='grey', linestyle='--')
    if sig_aleat is not None:
        mean_al = [np.mean(sig_aleat[:, li]) for li in range(OUTPUT_SEQ_LEN)]
        mean_ep = [np.mean(sig_epist[:, li]) for li in range(OUTPUT_SEQ_LEN)]
        axes[2].plot(leads, mean_al, marker='^', linewidth=2, color='darkorange', label='Aleatoric')
        axes[2].plot(leads, mean_ep, marker='v', linewidth=2, color='steelblue', label='Epistemic')
        axes[2].set_xlabel('Lead (months)'); axes[2].set_ylabel('σ (°C)')
        axes[2].set_title(f'Uncertainty Decomposition — {tag}')
        axes[2].legend(); axes[2].grid(True, alpha=0.3)
        rmse_per_lead = [
            float(np.sqrt(np.mean((preds[:, li] - targets[:, li])**2)))
            for li in range(OUTPUT_SEQ_LEN)
        ]
        total_unc = [np.sqrt(a**2 + e**2) for a, e in zip(mean_al, mean_ep)]
        axes[3].plot(leads, rmse_per_lead, marker='o', linewidth=2, label='RMSE', color='black')
        axes[3].plot(leads, total_unc, marker='s', linewidth=2, label='σ_total', color='purple')
        axes[3].set_xlabel('Lead (months)'); axes[3].set_ylabel('°C')
        axes[3].set_title(f'Spread-Skill — {tag}')
        axes[3].legend(); axes[3].grid(True, alpha=0.3)
    plt.tight_layout()
    path = FIGURE_DIR / fname
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f' → Predictability plot: {path}')


# ============================================================
# Progressive Fine-tuning
# ============================================================
def set_trainable(module, requires_grad):
    for param in module.parameters():
        param.requires_grad = requires_grad


def configure_phase(model, phase):
    if phase == 1:
        print(" Phase 1: decoder + head + season_enc only")
        set_trainable(model.spatial, False)
        set_trainable(model.encoder, False)
        set_trainable(model.season_enc, True)
        set_trainable(model.decoder, True)
        set_trainable(model.head, True)
        model.query_embed.requires_grad = True
    elif phase == 2:
        print(" Phase 2: unfreeze encoder + season_enc")
        set_trainable(model.spatial, False)
        set_trainable(model.encoder, True)
        set_trainable(model.season_enc, True)
        set_trainable(model.decoder, True)
        set_trainable(model.head, True)
        model.query_embed.requires_grad = True
    elif phase == 3:
        print(" Phase 3: full fine-tuning")
        set_trainable(model.spatial, True)
        set_trainable(model.encoder, True)
        set_trainable(model.season_enc, True)
        set_trainable(model.decoder, True)
        set_trainable(model.head, True)
        model.query_embed.requires_grad = True


def build_optimizer(model, base_lr, phase):
    if phase == 1:
        return torch.optim.AdamW([
            {'params': model.season_enc.parameters(), 'lr': base_lr * 0.5},
            {'params': model.decoder.parameters(), 'lr': base_lr},
            {'params': model.head.parameters(), 'lr': base_lr},
            {'params': [model.query_embed], 'lr': base_lr},
        ], weight_decay=1e-3)
    elif phase == 2:
        return torch.optim.AdamW([
            {'params': model.encoder.parameters(), 'lr': base_lr * 0.5},
            {'params': model.season_enc.parameters(), 'lr': base_lr * 0.5},
            {'params': model.decoder.parameters(), 'lr': base_lr},
            {'params': model.head.parameters(), 'lr': base_lr},
            {'params': [model.query_embed], 'lr': base_lr},
        ], weight_decay=1e-3)
    elif phase == 3:
        return torch.optim.AdamW([
            {'params': model.spatial.parameters(), 'lr': base_lr * 0.1},
            {'params': model.encoder.parameters(), 'lr': base_lr * 0.3},
            {'params': model.season_enc.parameters(), 'lr': base_lr * 0.5},
            {'params': model.decoder.parameters(), 'lr': base_lr},
            {'params': model.head.parameters(), 'lr': base_lr},
            {'params': [model.query_embed], 'lr': base_lr},
        ], weight_decay=1e-3)


def early_stopping_step(val_loss, best_loss, patience_counter, patience, min_delta=1e-4):
    improved = val_loss < (best_loss - min_delta)
    if improved:
        best_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
    return best_loss, patience_counter, patience_counter >= patience, improved


def progressive_finetune_auto(model, train_loader, val_loader, checkpoint_name, writer=None):
    phases = [
        {'phase': 1, 'lr': 1e-4, 'patience': 2, 'max_epochs': 10},
        {'phase': 2, 'lr': 5e-5, 'patience': 2, 'max_epochs': 10},
        {'phase': 3, 'lr': 1e-5, 'patience': 3, 'max_epochs': 15},
    ]
    global_best = np.inf
    ckpt = CHECKPOINT_DIR / f"best_{checkpoint_name}.pt"

    for stage in phases:
        phase = stage['phase']
        lr = stage['lr']
        patience = stage['patience']
        max_epochs = stage['max_epochs']
        print(f"\n === Phase {phase} | lr={lr} | patience={patience} ===")

        configure_phase(model, phase)
        optimizer = build_optimizer(model, lr, phase)

        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
                for p in module.parameters():
                    p.requires_grad = False

        best_val = np.inf
        patience_counter = 0

        for ep in range(max_epochs):
            model.train()
            if hasattr(train_loader.dataset, 'set_epoch'):
                train_loader.dataset.set_epoch(ep)
                
            tr_losses = []
            for X, y, months in tqdm(train_loader, desc=f"Phase {phase} Epoch {ep+1}", leave=False):
                X, y, months = X.to(DEVICE), y.to(DEVICE), months.to(DEVICE)
                optimizer.zero_grad()
                mean, log_var = model(X, months)
                loss = gaussian_nll_loss(mean, log_var, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                tr_losses.append(loss.item())

            model.eval()
            val_losses = []
            with torch.no_grad():
                for X, y, months in val_loader:
                    X, y, months = X.to(DEVICE), y.to(DEVICE), months.to(DEVICE)
                    if DEVICE == 'cuda':
                        with autocast(device_type='cuda'):
                            mean, log_var = model(X, months)
                            val_losses.append(gaussian_nll_loss(mean, log_var, y).item())
                    else:
                        mean, log_var = model(X, months)
                        val_losses.append(gaussian_nll_loss(mean, log_var, y).item())

            val_nll = np.mean(val_losses)
            best_val, patience_counter, stop, improved = early_stopping_step(
                val_nll, best_val, patience_counter, patience
            )

            if improved and val_nll < global_best:
                global_best = val_nll
                torch.save(model.state_dict(), ckpt)
                
            if writer and ENABLE_TENSORBOARD:
                writer.add_scalar(f'{checkpoint_name}/phase{phase}_train_loss', np.mean(tr_losses), ep)
                writer.add_scalar(f'{checkpoint_name}/phase{phase}_val_loss', val_nll, ep)

            print(f" Ep {ep+1:02d} | Train={np.mean(tr_losses):.4f} | "
                  f"Val={val_nll:.4f} | "
                  f"{'↑ improved' if improved else f'no improve ({patience_counter}/{patience})'}")

            if stop:
                print(f" → Early stopping triggered (phase {phase})")
                break

        print(f" → Phase {phase} done | best Val NLL={best_val:.4f}")

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False))
    print(f"\n ✅ Best model restored (global Val NLL={global_best:.4f})")
    return model, global_best


# ============================================================
# Ensemble Weighting — IMPROVED: memory-efficient
# ============================================================
def compute_ensemble_weights(fine_tuned_models, temperature=0.5):
    """Compute softmax weights from validation NLL."""
    nlls = np.array([m['finetune_val_nll'] for m in fine_tuned_models])
    nlls = (nlls - nlls.min()) / (nlls.std() + 1e-6)
    log_w = -nlls / temperature
    log_w -= log_w.max()  # Numerical stability
    weights = np.exp(log_w)
    return weights / weights.sum()


def compute_ensemble_predictions(predictions_list, weights):
    """Memory-efficient weighted ensemble: incremental accumulation."""
    if len(predictions_list) == 0:
        return None
    ensemble = np.zeros_like(predictions_list[0], dtype=np.float32)
    for pred, w in zip(predictions_list, weights):
        ensemble += w * pred
    return ensemble


# ============================================================
# Export Model for Inference — IMPROVED: dynamic dimensions
# ============================================================
def export_model_for_inference(model, output_path):
    model.eval()
    scripted_model = torch.jit.script(model)
    scripted_model.save(output_path)
    print(f"✓ Model exported to {output_path}")


# ============================================================
# Main Pipeline — HYBRID: all improvements integrated
# ============================================================
def main():
    print('=' * 90)
    print('IOD PREDICTION — HYBRID TRANSFER LEARNING (Best of Both Worlds)')
    print('=' * 90)
    print(f"Device: {DEVICE} | NUM_WORKERS: {NUM_WORKERS} | Scheduler: {SCHEDULER_TYPE}")
    if ENABLE_TENSORBOARD:
        print(f"TensorBoard: {TENSORBOARD_DIR.resolve()}")

    # Initialize TensorBoard writer
    writer = SummaryWriter(log_dir=str(TENSORBOARD_DIR / "iod_experiment")) if ENABLE_TENSORBOARD else None

    # ----------------------------------------------------------
    # STAGE 0: Prepare observations + shared normalization anchor
    # ----------------------------------------------------------
    print('\n[STAGE 0] Preparing observations...')
    ds_obs_full = load_obs_data(iod_only=False)
    ds_sst_obs = load_obs_data(iod_only=True)

    iod_full = calculate_monthly_dmi(
        ds_sst_obs['sst'].sel(time=slice(str(OBS_START), str(OBS_END))),
        base_period=CLIMATOLOGY_PERIOD
    )
    iod_ft_ref = iod_full.sel(time=slice(str(FT_START), str(FT_END)))
    iod_mean = float(iod_ft_ref.mean())
    iod_std = float(iod_ft_ref.std()) + 1e-6
    iod_norm = (iod_full - iod_mean) / iod_std
    print(f' IOD normalization (1960-1990): mean={iod_mean:.4f}, std={iod_std:.4f}')

    print(f'\n [Preprocessing] Computing OBS anchor climatology '
          f'({CLIMATOLOGY_PERIOD.start}–{CLIMATOLOGY_PERIOD.stop})')
    clim_obs, std_obs = compute_monthly_climatology(ds_obs_full, CLIMATOLOGY_PERIOD)
    set_obs_anchor(clim_obs, std_obs)

    ds_obs_std = apply_preprocessing(ds_obs_full, clim_obs, std_obs)
    da_obs, obs_channels = to_channel_da(ds_obs_std, ALL_VARS)
    
    ft_dataset = IODDataset(da_obs, iod_norm, obs_channels,
                            target_start_year=FT_START, target_end_year=FT_END,
                            augment=True, noise_scale=0.005, noise_decay_epoch=PRETRAIN_EPOCHS)
    test_dataset = IODDataset(da_obs, iod_norm, obs_channels,
                              target_start_year=TEST_START, target_end_year=TEST_END)

    print(f' Fine-tuning samples: {len(ft_dataset)}')
    print(f' Test samples: {len(test_dataset)}')

    # ----------------------------------------------------------
    # STAGE 1: Pre-train on CMIP6 models + ZERO-SHOT DIAGNOSTIC
    # ----------------------------------------------------------
    print('\n[STAGE 1] Pre-training on CMIP6 historical simulations...')
    model_files_dict = get_cmip6_model_files('model_1x1/')
    model_names = list(model_files_dict.keys())[:31]
    
    member_results = []
    
    for idx, model_name in enumerate(tqdm(model_names, desc="Pre-training models")):
        print(f'\n--- Pre-training model {idx+1}/{len(model_names)}: {model_name} ---')
        try:
            ds_cmip6 = load_single_cmip6_model('model_1x1/', model_name, model_files_dict)
            if ds_cmip6 is None:
                continue
        except Exception as e:
            print(f' ✗ Skipping: {e}')
            continue
        
        iod_cmip6 = calculate_monthly_dmi(ds_cmip6['sst'], base_period=CLIMATOLOGY_PERIOD)
        iod_c_pre94 = iod_cmip6.sel(time=slice(str(CMIP6_START), '1993'))
        c_mean = float(iod_c_pre94.mean())
        c_std = float(iod_c_pre94.std()) + 1e-6
        iod_c_norm = (iod_cmip6 - c_mean) / c_std
        
        clim_c, std_c = compute_monthly_climatology(ds_cmip6, CLIMATOLOGY_PERIOD)
        ds_c_std = apply_preprocessing_shared(ds_cmip6, clim_c, std_c, use_anchor=True)
        da_cmip6, cmip_channels = to_channel_da(ds_c_std, ALL_VARS)
        
        shared_channels = [c for c in obs_channels if c in cmip_channels]
        if 'sst' not in shared_channels:
            print(f' ✗ No SST in shared channels, skipping')
            del ds_cmip6, da_cmip6
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            continue
        
        channel_indices = [obs_channels.index(c) for c in shared_channels]
        cmip_indices = [cmip_channels.index(c) for c in shared_channels]
        da_cmip6_shared = da_cmip6.isel(channel=cmip_indices)
        
        # IMPROVED: Channel alignment assertion
        assert shared_channels == [obs_channels[i] for i in channel_indices], \
            f"Channel ordering mismatch: expected {shared_channels}"
        
        assert da_cmip6_shared.shape[1] == len(shared_channels), \
            "Channel dimension mismatch" 
        
        train_c = IODDataset(da_cmip6_shared, iod_c_norm, shared_channels)
        val_c = IODDataset(da_cmip6_shared, iod_c_norm, shared_channels,
                           target_start_year=PRETRAIN_VAL_START)
        
        train_loader = create_safe_dataloader(train_c, BATCH_SIZE, shuffle=True)
        val_loader = create_safe_dataloader(val_c, BATCH_SIZE, shuffle=False)
        
        model = SpatioTemporalIOD(len(shared_channels)).to(DEVICE)
        
        # 🎯 ZERO-SHOT DIAGNOSTIC: Evaluate on obs BEFORE fine-tuning
        ft_eval_loader = create_safe_dataloader(ft_dataset, BATCH_SIZE, shuffle=False)
        
        print(f' Zero-shot NLL on fine-tuning window: {zero_shot_nll:.4f} (informational)')
        
        model, _ = train_model(model, train_loader, val_loader,
                               PRETRAIN_EPOCHS, 1e-3,
                               f'pretrain_{model_name}', is_finetune=False, 
                               writer=writer, resume=RESUME_TRAINING)
        zero_shot_nll = evaluate_zero_shot(model, ft_eval_loader, channel_indices, DEVICE)
       
        member_results.append({
            'name': model_name,
            'pretrain_ckpt': CHECKPOINT_DIR / f'best_pretrain_{model_name}.pt',
            'channels': shared_channels,
            'channel_indices': channel_indices,
            'zero_shot_nll': zero_shot_nll,  # Store for filtering/analysis
        })
        
        # Aggressive cleanup
        del ds_cmip6, da_cmip6, model, train_loader, val_loader, train_c, val_c
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    
    print(f"Successfully pre-trained {len(member_results)} models.")
    
    # ----------------------------------------------------------
    # STAGE 2: Fine-tuning on observations (1960-1990)
    # ----------------------------------------------------------
    print('\n[STAGE 2] Fine-tuning on early observations (1960-1990)...')
    fine_tuned_models = []

    for member in tqdm(member_results, desc="Fine-tuning"):
        print(f'\n--- Fine-tuning {member["name"]} ---')
        da_obs_subset = da_obs.isel(channel=member['channel_indices'])
        ft_dataset_member = IODDataset(da_obs_subset, iod_norm, member['channels'],
                                       target_start_year=FT_START, target_end_year=FT_END,
                                       augment=True, noise_scale=0.005, 
                                       noise_decay_epoch=PRETRAIN_EPOCHS+35)  # Total FT epochs

        #indices = np.arange(len(ft_dataset_member))
        #split = int(0.8 * len(indices))
        #gap = SAMPLE_GAP
        #ft_train = torch.utils.data.Subset(ft_dataset_member, indices[:split - gap])
        #ft_val = torch.utils.data.Subset(ft_dataset_member, indices[split:])
        ft_train, ft_val = split_dataset_by_time(
        ft_dataset_member,
        train_end_year=1980,
        val_start_year=1981
        )

        
        ft_train_loader = create_safe_dataloader(ft_train, BATCH_SIZE, shuffle=True)
        ft_val_loader = create_safe_dataloader(ft_val, BATCH_SIZE, shuffle=False)

        model = SpatioTemporalIOD(len(member['channels'])).to(DEVICE)
        
        pretrain_ckpt = member['pretrain_ckpt']
        if pretrain_ckpt.exists():
            model.load_state_dict(torch.load(pretrain_ckpt, map_location=DEVICE, weights_only=False))
            print(f" ✓ Loaded pretrained weights from {pretrain_ckpt}")
        else:
            print(f" ⚠ Pretrained checkpoint not found: {pretrain_ckpt}")

        model, finetune_val_nll = progressive_finetune_auto(
            model, ft_train_loader, ft_val_loader, f'finetune_{member["name"]}', writer=writer
        )

        print(f' Post-finetune val NLL: {finetune_val_nll:.4f}')
        finetune_ckpt = CHECKPOINT_DIR / f'best_finetune_{member["name"]}.pt'
        torch.save(model.state_dict(), finetune_ckpt)

        fine_tuned_models.append({
            'name': member['name'],
            'model': model,
            'channels': member['channels'],
            'channel_indices': member['channel_indices'],
            'finetune_ckpt': finetune_ckpt,
            'zero_shot_nll': member.get('zero_shot_nll', 0.0),
            'finetune_val_nll': finetune_val_nll,
        })

    # ----------------------------------------------------------
    # STAGE 3: Test on modern observations (2000-2024)
    # ----------------------------------------------------------
    print(f'\n[STAGE 3] FINAL TEST EVALUATION ({TEST_START}-{TEST_END})')
    
    if not fine_tuned_models:
        print("No fine-tuned models available. Exiting.")
        return
        
    test_predictions = []
    test_s_al_list = []
    test_s_ep_list = []
    tgts = None
    test_times = None

    for member in fine_tuned_models:
        da_obs_subset = da_obs.isel(channel=member['channel_indices'])
        test_ds = IODDataset(da_obs_subset, iod_norm, member['channels'],
                             target_start_year=TEST_START, target_end_year=TEST_END)

        preds, tgts_m, times, s_al, s_ep, _ = run_inference(
            member['model'], test_ds, iod_mean, iod_std, return_uncertainty=True
        )
        test_predictions.append(preds)
        test_s_al_list.append(s_al)
        test_s_ep_list.append(s_ep)
        if tgts is None:
            tgts = tgts_m
            test_times = times
        print(f' {member["name"]} — test samples: {len(preds)}')

    # Trim to same length
    min_len = min(p.shape[0] for p in test_predictions)
    test_predictions = [p[:min_len] for p in test_predictions]
    test_s_al_list = [s[:min_len] for s in test_s_al_list]
    test_s_ep_list = [s[:min_len] for s in test_s_ep_list]
    tgts = tgts[:min_len]
    test_times = test_times[:min_len]

    weights = compute_ensemble_weights(fine_tuned_models, temperature=0.5)
    print(f'\n Softmax ensemble weights (post-finetune val NLL, τ=0.5):')
    for m, w in zip(fine_tuned_models, weights):
        print(f" {m['name']:25s} val_nll={m['finetune_val_nll']:.4f} w={w:.4f}")

    # Save ensemble weights
    with open(RESULTS_DIR / "ensemble_weights.pkl", "wb") as f:
        pickle.dump({'weights': weights, 'models': [m['name'] for m in fine_tuned_models]}, f)

    # IMPROVED: Memory-efficient ensemble prediction
    ensemble_test_preds = compute_ensemble_predictions(test_predictions, weights)
    ensemble_spread = np.std(np.stack(test_predictions), axis=0)
    weighted_s_al = compute_ensemble_predictions(test_s_al_list, weights)
    weighted_s_ep = compute_ensemble_predictions(test_s_ep_list, weights)
    total_uncertainty = np.sqrt(ensemble_spread**2 + weighted_s_al**2 + weighted_s_ep**2)
    #total_uncertainty = np.sqrt(ensemble_spread**2 + weighted_s_al**2)
    
    print_verification_table(
        ensemble_test_preds, tgts,
        tag=f'Ensemble (fine-tuned) | TEST {TEST_START}-{TEST_END}',
        sig_aleat=weighted_s_al, sig_epist=weighted_s_ep
    )

    save_timeseries_figure(
        test_times, tgts, ensemble_test_preds,
        tag='Transfer Ensemble (test)',
        fname='transfer_ensemble_test.png',
        sig_total=total_uncertainty
    )

    plot_predictability_vs_lead(
        ensemble_test_preds, tgts,
        tag=f'Test {TEST_START}-{TEST_END}',
        fname='predictability_test.png',
        sig_aleat=weighted_s_al, sig_epist=weighted_s_ep
    )

    # Export a sample model for inference — IMPROVED: dynamic dimensions
    if fine_tuned_models:
        sample_model = fine_tuned_models[0]['model']
        H, W = da_obs.shape[-2], da_obs.shape[-1]
        H_down = H // SPATIAL_DOWNSAMPLE_FACTOR
        W_down = W // SPATIAL_DOWNSAMPLE_FACTOR
        n_channels = len(fine_tuned_models[0]['channels'])
        sample_input = torch.randn(1, INPUT_WINDOW, n_channels, H_down, W_down).to(DEVICE)
        sample_months = torch.randint(0, 12, (1, INPUT_WINDOW)).to(DEVICE)
        export_model_for_inference(sample_model, RESULTS_DIR / "iod_model_exported.pt")


    # ----------------------------------------------------------
    # EXTRA: Validation period evaluation
    # ----------------------------------------------------------
    print(f'\n[EXTRA] Validation period ({FT_START}-{FT_END})')
    val_predictions = []
    val_s_al_list = []
    val_s_ep_list = []
    tgts_val = None
    val_times = None

    for member in fine_tuned_models:
        da_obs_subset = da_obs.isel(channel=member['channel_indices'])
        val_ds = IODDataset(da_obs_subset, iod_norm, member['channels'],
                            target_start_year=FT_START, target_end_year=FT_END)
        preds, tgts_v, times, s_al, s_ep, _ = run_inference(
            member['model'], val_ds, iod_mean, iod_std, return_uncertainty=True
        )
        val_predictions.append(preds)
        val_s_al_list.append(s_al)
        val_s_ep_list.append(s_ep)
        if tgts_val is None:
            tgts_val = tgts_v
            val_times = times

    min_len_val = min(p.shape[0] for p in val_predictions)
    val_predictions = [p[:min_len_val] for p in val_predictions]
    val_s_al_list = [s[:min_len_val] for s in val_s_al_list]
    val_s_ep_list = [s[:min_len_val] for s in val_s_ep_list]
    tgts_val = tgts_val[:min_len_val]
    val_times = val_times[:min_len_val]

    ensemble_val_preds = compute_ensemble_predictions(val_predictions, weights)
    val_spread = np.std(np.stack(val_predictions), axis=0)
    val_weighted_s_al = compute_ensemble_predictions(val_s_al_list, weights)
    val_weighted_s_ep = compute_ensemble_predictions(val_s_ep_list, weights)
    val_total_unc = np.sqrt(val_spread**2 + val_weighted_s_al**2 + val_weighted_s_ep**2)

    print_verification_table(
        ensemble_val_preds, tgts_val,
        tag=f'Ensemble (fine-tuned) | VAL {FT_START}-{FT_END}',
        sig_aleat=val_weighted_s_al, sig_epist=val_weighted_s_ep
    )

    save_timeseries_figure(
        val_times, tgts_val, ensemble_val_preds,
        tag='Transfer Ensemble (validation)',
        fname='transfer_ensemble_validation.png',
        sig_total=val_total_unc
    )

    # Close TensorBoard writer
    if writer:
        writer.close()

    print('\n' + '=' * 90)
    print('✅ PIPELINE COMPLETED SUCCESSFULLY')
    print(f' Checkpoints : {CHECKPOINT_DIR.resolve()}')
    print(f' Figures : {FIGURE_DIR.resolve()}')
    print(f' Results : {RESULTS_DIR.resolve()}')
    if ENABLE_TENSORBOARD:
        print(f' TensorBoard : {TENSORBOARD_DIR.resolve()}')
        print('   → Run: tensorboard --logdir runs/')
    print('=' * 90)

    def persistence_forecast(targets):
        pred = np.empty_like(targets)
        pred[:] = np.nan
        pred[1:] = targets[:-1]
        return pred
    
    baseline_preds = persistence_forecast(tgts)
    
    print_verification_table(
        baseline_preds, tgts,
        tag="Persistence baseline"
    )

if __name__ == '__main__':
    # Handle keyboard interrupt gracefully
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️ Interrupted by user. Cleaning up...")
        sys.exit(0)
    except BrokenPipeError:
        print("\n\n❌ Broken pipe error occurred.")
        print("   Try setting NUM_WORKERS=0: export NUM_WORKERS=0")
        sys.exit(1)